# Predict (by hand) - Extra Lab
* **Post-unload memory**: Expected to be **close to baseline** (within **~10-50 MB** drift due to CUDA Caching Allocator).
* **Accumulating tensor outputs**: Linear VRAM growth across iterations. This is **expected behaviour** (not a PyTorch bug) because references keep the tensors and computation graph alive.

### [Cell 1] Step 1: The Reload-Loop Baseline (Control Test)

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [1]:
import gc, torch
from transformers import AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


def reserved_mb():
  torch.cuda.synchronize()
  return torch.cuda.memory_reserved() / (1024**2)


def unload(model):
  del model
  gc.collect()
  torch.cuda.empty_cache()


samples = []
for i in range(5):
  model = AutoModelForCausalLM.from_pretrained(
      MODEL, torch_dtype=torch.float16, device_map="cuda"
  )
  after_load = reserved_mb()
  unload(model)
  after_unload = reserved_mb()
  samples.append({
      "cycle": i,
      "after_load_mb": round(after_load, 1),
      "after_unload_mb": round(after_unload, 1),
  })
  print(samples[-1])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

{'cycle': 2, 'after_load_mb': 5946.0, 'after_unload_mb': 5892.0}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'cycle': 3, 'after_load_mb': 5946.0, 'after_unload_mb': 5892.0}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'cycle': 4, 'after_load_mb': 5946.0, 'after_unload_mb': 5890.0}


### [Cell 2] Step 2: Introduce a Real Leak (On Purpose)

In [2]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda"
)
tok_ids = torch.randint(0, 1000, (1, 64)).to("cuda")

leaked_outputs = []  # يحتفظ بالـ Tensors في الذاكرة
leak_samples = []
for i in range(20):
  # تم تعمد حذف torch.no_grad() للاحتفاظ بـ Autograd Graph
  out = model(tok_ids)
  leaked_outputs.append(out.logits)
  leak_samples.append({"iter": i, "reserved_mb": round(reserved_mb(), 1)})
  if i % 5 == 0:
    print(leak_samples[-1])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'iter': 0, 'reserved_mb': 5972.0}
{'iter': 5, 'reserved_mb': 6424.0}
{'iter': 10, 'reserved_mb': 6876.0}
{'iter': 15, 'reserved_mb': 7328.0}


### [Cell 3] Step 3: The Leak Detector Script

In [3]:
import numpy as np


def detect_leak(samples_mb, slope_threshold_mb_per_iter=1.0):
  """samples_mb: list of reserved-memory readings, one per iteration."""
  x = np.arange(len(samples_mb))
  y = np.array(samples_mb)
  slope, intercept = np.polyfit(x, y, 1)
  leaking = slope > slope_threshold_mb_per_iter
  return {
      "slope_mb_per_iter": round(float(slope), 3),
      "threshold_mb_per_iter": slope_threshold_mb_per_iter,
      "leaking": bool(leaking),
      "n_samples": len(samples_mb),
  }


leak_result = detect_leak([s["reserved_mb"] for s in leak_samples])
print("Leak Result:", leak_result)
assert leak_result["leaking"], (
    "expected the Step 2 loop to be flagged as leaking"
)

Leak Result: {'slope_mb_per_iter': 90.412, 'threshold_mb_per_iter': 1.0, 'leaking': True, 'n_samples': 20}


### [Cell 4] Step 4: Fix the Leak and Reconfirm

In [4]:
# تفريغ النموذج التالف وإعادة تحميل نظيف
unload(model)
del leaked_outputs
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda"
)

fixed_samples = []
# الحل: استخدام torch.no_grad وعدم تخزين الـ Tensors
with torch.no_grad():
  for i in range(20):
    out = model(tok_ids)
    _ = out.logits.sum().item()  # استخراج قيمة عددية بدون حفظ Tensor
    fixed_samples.append({"iter": i, "reserved_mb": round(reserved_mb(), 1)})

fixed_result = detect_leak([s["reserved_mb"] for s in fixed_samples])
print("Fixed Result:", fixed_result)
assert not fixed_result["leaking"], (
    "still leaking after the fix -- check both causes were removed"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fixed Result: {'slope_mb_per_iter': -0.0, 'threshold_mb_per_iter': 1.0, 'leaking': False, 'n_samples': 20}


In [7]:
import json
import numpy as np

with open("leak_report.json") as f:
  r = json.load(f)

# التحقق من وجود الحقول الأساسية
assert "leaky_samples" in r and "fixed_samples" in r, "Missing raw samples"

# إعادة حساب الميل بشكل مستقل
x_leak = np.arange(len(r["leaky_samples"]))
slope_leak, _ = np.polyfit(x_leak, r["leaky_samples"], 1)

x_fixed = np.arange(len(r["fixed_samples"]))
slope_fixed, _ = np.polyfit(x_fixed, r["fixed_samples"], 1)

print(f"Independent Leaky Slope: {slope_leak:.3f} MB/iter")
print(f"Independent Fixed Slope: {slope_fixed:.3f} MB/iter")

# شروط الاجتياز: التسريب يتجاوز 5 MB/iter والإصلاح يظل مسطحاً دون 1 MB/iter
assert slope_leak >= 5.0, (
    f"Leaky run did not leak significantly: {slope_leak:.2f} MB/iter"
)
assert slope_fixed < 1.0, (
    f"Fixed run still has upward drift: {slope_fixed:.2f} MB/iter"
)
assert r["leaky_run"]["leaking"] is True, "leaky_run flag mismatch"
assert r["fixed_run"]["leaking"] is False, "fixed_run flag mismatch"

print("\n" + "=" * 30)
print("GREEN CHECK: PASS")
print("=" * 30)

Independent Leaky Slope: 90.412 MB/iter
Independent Fixed Slope: -0.000 MB/iter

GREEN CHECK: PASS


### [Cell 5] Step 5: Generate Report and Verification

In [8]:
import json, os
from google.colab import files

report = {
    "reload_loop_baseline": samples,
    "leaky_run": leak_result,
    "fixed_run": fixed_result,
    "leaky_samples": [s["reserved_mb"] for s in leak_samples],
    "fixed_samples": [s["reserved_mb"] for s in fixed_samples],
}

with open("leak_report.json", "w") as f:
  json.dump(report, f, indent=2)

print("Saved leak_report.json successfully!")
files.download("leak_report.json")

Saved leak_report.json successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>